# FMO inference demo — nine-feature structured report from the deployed checkpoints

Companion notebook to *Deep learning feature reports as a drafting assistant for single-operator appendiceal ultrasound* (code: https://github.com/ER-UIS/FMS-Appendicitis-US; images, labels and predictions: https://doi.org/10.6084/m9.figshare.29222042). The nine deployed weight files are available from the corresponding authors on request; place them, with `checkpoints.json` (= `config/deployed_checkpoints.json`), in the checkpoint folder set in section 1.

What it does:
1. verifies the nine deployed FMO weight files against `checkpoints.json` (MD5);
2. rebuilds the architecture (InceptionResNetV2, ImageNet-transfer, 224 × 224, two-class softmax) and loads each checkpoint;
3. scores every image of a patient folder and applies the patient-level rule used in the paper (class of the most confident image-level prediction);
4. assembles the nine-feature structured report, compares it with the original clinical report if one is supplied (discrepancy count; substandard = ≥4 of 9, discarded = ≥6), and, optionally, lists the alerts of the feature-selective rule of an earlier version of this work (intraluminal gas, periappendiceal fluid, regional blood flow; not reported in the accompanying article).

Requirements: `pip install "tensorflow>=2.8,<2.16" numpy pandas pillow openpyxl` (Keras 2 API; on TF ≥2.16 install `tf-keras` and set `TF_USE_LEGACY_KERAS=1`). CPU is sufficient (about 1–2 s per image per feature).

**Research use only. Appendiceal wall condition (AWC) is withheld from automated output (see the paper); it is computed here only to reproduce research metrics and is marked accordingly.**

In [ ]:
import os, json, hashlib, glob
import numpy as np, pandas as pd

# ---- paths: edit these three lines ----
CKPT_DIR = "FMO_deployed_checkpoints"      # folder with the nine *_weights.h5 files + checkpoints.json (from figshare)
IMAGE_ROOT = "sample_images"               # one sub-folder per patient: IMAGE_ROOT/<App_No>/*.jpg
REPORT_XLSX = None                         # optional: original clinical reports, columns App_No, AODC, CC, ... (e.g. data/Test_data_3214.xlsx)

manifest = json.load(open(os.path.join(CKPT_DIR, "checkpoints.json"), encoding="utf-8"))
CK = manifest["checkpoints"]
pd.DataFrame(CK)[["index", "feature_code", "feature", "deployed_epoch", "weights_file", "positive_label", "deployable"]]

## 1. Verify the weight files

In [ ]:
def md5(path, chunk=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

ok = True
for c in CK:
    p = os.path.join(CKPT_DIR, c["weights_file"])
    if not os.path.exists(p):
        print("MISSING", c["weights_file"]); ok = False; continue
    m = md5(p)
    print(f"{c['feature_code']:5s} {c['weights_file']:38s} {'OK' if m == c['md5'] else 'MD5 MISMATCH'}")
    ok &= (m == c["md5"])
assert ok, "download the nine weight files from the figshare record into CKPT_DIR" 

## 2. Rebuild the architecture and load the checkpoints

In [ ]:
import tensorflow as tf
print("TensorFlow", tf.__version__)

def build_model(num_classes=2, image_size=224):
    base = tf.keras.applications.InceptionResNetV2(include_top=False, weights=None,
                                                   input_shape=(image_size, image_size, 3), pooling="avg")
    out = tf.keras.layers.Dense(num_classes, activation="softmax", use_bias=True)(base.output)
    return tf.keras.Model(inputs=base.input, outputs=out)

models = {}
for c in CK:
    m = build_model()
    path = os.path.join(CKPT_DIR, c["weights_file"])
    try:
        m.load_weights(path)
    except Exception as e:            # layer names differ in this Keras build -> match by name
        print(c["feature_code"], "falling back to by_name loading:", type(e).__name__)
        m.load_weights(path, by_name=True, skip_mismatch=False)
    models[c["feature_code"]] = m
print("loaded", len(models), "models")

## 3. Image preprocessing and the patient-level rule

In [ ]:
from PIL import Image

IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp")

def load_images(folder, size=224):
    names = sorted(n for n in os.listdir(folder) if n.lower().endswith(IMG_EXT))
    arr = np.stack([np.asarray(Image.open(os.path.join(folder, n)).convert("RGB").resize((size, size)), dtype=np.float32) / 255.0
                    for n in names]) if names else None
    return names, arr

def predict_feature(c, arr, names):
    """Patient-level prediction of one feature: class of the single most confident image-level prediction
    (maximum softmax probability over all images and both classes), as in the original pipeline."""
    probs = models[c["feature_code"]].predict(arr, batch_size=16, verbose=0)      # (n_images, 2)
    i_img, i_cls = np.unravel_index(np.argmax(probs), probs.shape)
    labels = c["class_index_to_label"]
    pos = c["positive_class_index"]
    return {"feature": c["feature_code"], "label": labels[str(i_cls)], "present": bool(i_cls == pos),
            "p_positive": float(probs[i_img, pos]), "confidence": float(probs[i_img, i_cls]),
            "image": names[i_img], "n_images": len(names), "deployable": c.get("deployable", True)}

def structured_report(patient_folder):
    names, arr = load_images(patient_folder)
    if arr is None:
        raise FileNotFoundError(f"no images in {patient_folder}")
    rows = [predict_feature(c, arr, names) for c in CK]
    df = pd.DataFrame(rows).set_index("feature")
    df.loc["AWC", "label"] = df.loc["AWC", "label"] + "  [withheld from automated output]"
    return df

## 4. Run on sample patients

Place de-identified images as `sample_images/<App_No>/*.jpg` (all diagnostic-quality images of the examination; every feature classifier sees the same images). If no folder exists, a synthetic image is used as a smoke test of the pipeline (its output is meaningless).

In [ ]:
patients = sorted(d for d in glob.glob(os.path.join(IMAGE_ROOT, "*")) if os.path.isdir(d))
if not patients:
    os.makedirs(os.path.join(IMAGE_ROOT, "synthetic_smoke_test"), exist_ok=True)
    Image.fromarray((np.random.rand(480, 640, 3) * 255).astype(np.uint8)).save(os.path.join(IMAGE_ROOT, "synthetic_smoke_test", "noise.jpg"))
    patients = [os.path.join(IMAGE_ROOT, "synthetic_smoke_test")]
    print("No sample images found - running a synthetic smoke test only.")

reports = {}
for p in patients:
    pid = os.path.basename(p)
    reports[pid] = structured_report(p)
    print(f"\n=== Patient {pid} ({int(reports[pid]['n_images'].iloc[0])} images) ===")
    display(reports[pid][["label", "present", "p_positive", "confidence", "image"]])

## 5. Compare with the original clinical report (optional)

The study end point (a study-defined discrepancy burden) counts, per report, the features on which the automated report disagrees with the reference (the original clinical report in primary analysis 1). A report is *substandard* with ≥4 of 9 discrepancies and *discarded* with ≥6. The feature-selective alert rule of an earlier version of this work (not reported in the accompanying article) raises an alert only when the model disagrees with the report on intraluminal gas (GAC), periappendiceal fluid (PFCC), or regional blood flow (BFC).

In [ ]:
ALERT_FEATURES = ["GAC", "PFCC", "BFC"]
POS = {c["feature_code"]: c["positive_label"] for c in CK}
CODES = [c["feature_code"] for c in CK]

if REPORT_XLSX and os.path.exists(REPORT_XLSX):
    ref = pd.read_excel(REPORT_XLSX, dtype={"App_No": str}).set_index("App_No")
    for pid, rep in reports.items():
        if pid not in ref.index:
            print(pid, "not in the report file"); continue
        report_present = {k: (str(ref.loc[pid, k]) == POS[k]) for k in CODES}
        model_present = rep["present"].to_dict()
        disc = [k for k in CODES if model_present[k] != report_present[k]]
        alerts = [k for k in disc if k in ALERT_FEATURES]
        print(f"Patient {pid}: {len(disc)} discrepant feature(s) {disc} -> "
              f"{'DISCARDED' if len(disc) >= 6 else 'SUBSTANDARD' if len(disc) >= 4 else 'acceptable'}; "
              f"feature-selective alerts: {alerts or 'none'}")
else:
    print("REPORT_XLSX not set - skipping the comparison. Use data/Test_data_3214.xlsx from the repository for the internal test cohort.")

## Notes

* The patient-level rule above reproduces the pipeline that generated all published predictions (`fms/predict.py` in the repository). With softmax outputs it is equivalent to taking the class of the image with the largest margin.
* Checkpoint epochs: AODC 14, CC 41, CAE 4, AWC 4, GAC 9, FAC 9, PFCC 9, MSC 46, BFC 4 (Supplementary Table 2).
* Numerical outputs can differ in the last digits between TensorFlow versions and hardware; class decisions are stable except for probabilities very close to 0.5.